# Download the Perturb-seq datasets

Downloads every dataset in [`datasets.tsv`](datasets.tsv) into `perturbseq/raw/<dataset>/` under the data root: cell-level counts for the eight datasets of [`2.train_test`](../2.train_test/), and pseudobulk counts for Marson T cells (its cell-level files, 1.74 TB, are not used).
PIE's processed Replogle–Nadig arrays go into `pie_benchmark/`.
Each row of the table lists its files in three space-separated columns, in the same order: `file` (the name the authors released it under), `url`, and `checksum` (`md5:` or `sha256:`, `-` where none was recorded).
A file already on disk is skipped. A new file is checked against its checksum, and its sha256 is printed when none was recorded.

In [ ]:
import sys

import numpy as np
import pandas as pd

sys.path.append("../utils")
from download import download
from paths import data_dir

datasets = pd.read_csv("datasets.tsv", sep="\t")
datasets

In [ ]:
for row in datasets.itertuples():
    out = data_dir("pie_benchmark") if row.dataset == "pie_replogle_nadig" else data_dir("perturbseq", "raw", row.dataset)
    for name, url, checksum in zip(row.file.split(), row.url.split(), row.checksum.split()):
        path = out / name
        if path.exists():
            continue
        path.parent.mkdir(parents=True, exist_ok=True)
        sha = download(url, path, checksum)
        print(row.dataset, name, path.stat().st_size, "bytes", f"sha256:{sha}" if sha else "")